# Inverse Dynamics:
# Move with PD control

#### 1. Load scene

In [1]:
import os
import sys
import numpy as np
import time

import mujoco

sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *

In [2]:
xml_path = '../asset/panda_scene.xml'
# xml_path = '../asset/ur_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

# set collision group visualize False
# collision group: 1
target_group = 0
new_rgba = [0.0, 0.0, 0.0, 0.0] 
for i in range(model.ngeom):
    if model.geom_group[i] == target_group:
        model.geom_rgba[i] = new_rgba
mujoco.mj_forward(model, data)

In [3]:
""" GO TO INITIAL QPOS """
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data) # calculate initial dynamics

In [4]:
""" UTIL """
def euler2rmat(rpy_list):
    roll, pitch, yaw = rpy_list
    R_x = np.array([[1, 0, 0],
                    [0, np.cos(roll), -np.sin(roll)],
                    [0, np.sin(roll), np.cos(roll)]])
    
    R_y = np.array([[np.cos(pitch), 0, np.sin(pitch)],
                    [0, 1, 0],
                    [-np.sin(pitch), 0, np.cos(pitch)]])
    
    R_z = np.array([[np.cos(yaw), -np.sin(yaw), 0],
                    [np.sin(yaw), np.cos(yaw), 0],
                    [0, 0, 1]])
    
    R = R_z @ R_y @ R_x
    return R

def rmat2euler(R):
    sy = np.sqrt(R[0,0] * R[0,0] +  R[1,0] * R[1,0])
    singular = sy < 1e-6

    if not singular:
        roll = np.arctan2(R[2,1] , R[2,2])
        pitch = np.arctan2(-R[2,0], sy)
        yaw = np.arctan2(R[1,0], R[0,0])
    else:
        roll = np.arctan2(-R[1,2], R[1,1])
        pitch = np.arctan2(-R[2,0], sy)
        yaw = 0

    return np.array([roll, pitch, yaw])

""" POSITION-ROTATION RATIO """
pos_rot_ratio = 0.57 # (1/100) / (3.14/180)
# 1 centi-meter per 1 degree
# multiply to rotation error to get equivalent position error

### 2. PD Control
- Get position error & difference
- Get torque with P&D gain

In [5]:
""" GET SITE POSITION """
site_names = get_site_names(model, data)
print(site_names) # 'eef_site' , id 2
eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
eef_site_pos = data.site_xpos[eef_site_id]
print("eef_site_pos: ", eef_site_pos)

""" GET SITE ROTATION  """
eef_site_rmat = data.site_xmat[eef_site_id]
eef_site_rmat = eef_site_rmat.reshape(3, 3)
print("eef_site_rmat: ", eef_site_rmat)

['right_center', 'eef_site']
eef_site_pos:  [ 3.92404417e-01 -6.56356925e-34  4.58585349e-01]
eef_site_rmat:  [[ 9.59410820e-01  2.82012194e-01  1.57009246e-16]
 [ 2.82012194e-01 -9.59410820e-01 -9.24446373e-33]
 [ 1.50636369e-16  4.42785220e-17 -1.00000000e+00]]


In [6]:
""" TARGET SITE POSITION """

target_position_up = eef_site_pos + np.array([0, 0, 0.2])
target_position_down = eef_site_pos + np.array([0, 0, -0.2])

""" TARGET SITE ROTATION """
target_euler_up = np.array([0, 0, 0]) # 180 degree rotation around x-axis
target_euler_down = np.array([3.14, 0, 0]) # no rotation
target_rmat_up = euler2rmat(target_euler_up)
target_rmat_down = euler2rmat(target_euler_down)

""" PD GAIN """
Kp = 0.1  # Proportional gain
Kd = 5.0   # Derivative gain   

In [7]:
""" POSITIONAL ERROR """
# position error
pos_error = target_position_up - eef_site_pos
# orientation error
rmat_error = target_rmat_up @ eef_site_rmat.T
euler_error = rmat2euler(rmat_error) * pos_rot_ratio


""" VELOCITY ERROR """
dt = model.opt.timestep
# velocity error
pos_error_diff = (target_position_up - eef_site_pos) / dt
# orientation velocity error
euler_error_diff = (euler_error * pos_rot_ratio) / dt


""" TORQUE COMMAND """
pos_command = Kp * pos_error + Kd * pos_error_diff
rot_command = Kp * euler_error + Kd * euler_error_diff
cartesian_command = np.hstack((pos_command, rot_command))
print("cartesian_command: ", cartesian_command)

cartesian_command:  [ 0.00000000e+00  0.00000000e+00  5.00020000e+02 -2.55193770e+03
 -1.27539709e-13  2.32231088e+02]


In [8]:
""" GET JACOBIAN """
jacobian_p = np.zeros((3, model.nv))
jacobian_r = np.zeros((3, model.nv))
mujoco.mj_jacSite(model, data, jacobian_p, jacobian_r, eef_site_id) # no rotation target 
jacobian = np.vstack((jacobian_p, jacobian_r))
torque_command = jacobian.T @ cartesian_command
print("torque_command: ", torque_command)

torque_command:  [  232.23108814  -196.21005662  1427.26606135   235.760593
 -2417.11262004    44.00176     -232.23108814]


In [9]:
""" COMPENSATION TORQUE """
gravity_compensation = np.zeros(model.nv)
mujoco.mj_inverse(model, data)
gravity_compensation = data.qfrc_inverse - data.qfrc_applied    

#### 3. Main Loop
- Calculate desired torque every loop
- Clip to get stable torque

In [10]:
""" MAIN LOOP """
viewer = MUJOCOGLVIEWER(model, data)
mujoco.mj_resetData(model, data)
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

# save positional error for differentiation
prev_pos_error = pos_error
prev_rot_error = euler_error 
prev_rmat_error = rmat_error

pos_flag = True

while viewer.is_alive():
    glfw.poll_events()
    if glfw.get_key(viewer.windows[0], glfw.KEY_SPACE) == glfw.PRESS:
        pos_flag = not pos_flag
        if pos_flag:
            print("switch to up")
        else:
            print("switch to down")
    if pos_flag:
        target_position = target_position_up
        target_euler = target_euler_up
    else:
        target_position = target_position_down
        target_euler = target_euler_down
    target_rmat = euler2rmat(target_euler)

    # 1. get current site position
    eef_site_pos = data.site_xpos[eef_site_id]
    eef_site_rmat = data.site_xmat[eef_site_id].reshape(3, 3)
    eef_site_euler = rmat2euler(eef_site_rmat)

    # 2. calculate positional error
    # position 
    pos_error = target_position - eef_site_pos
    pos_error_diff = (pos_error - prev_pos_error) / dt
    prev_pos_error = pos_error
    # # orientation 1. use rotation matrix 
    # rot_diff = target_rmat @ eef_site_rmat.T
    # rot_error = rmat2euler(rot_diff) * pos_rot_ratio
    # orientation 2. use euler angle
    rot_error = (target_euler - eef_site_euler) * pos_rot_ratio
    
    # rot_error_diff = rmat2euler(rot_diff @ prev_rmat_error.T) / dt
    rot_error_diff = (rot_error - prev_rot_error) / dt
    prev_rot_error = rot_error
    # prev_rmat_error = rot_diff
    

    # 3. calculate torque command using PD control
    pos_command = Kp * pos_error + Kd * pos_error_diff
    rot_command = Kp * rot_error + Kd * rot_error_diff
    pos_command_clipped = np.clip(pos_command, -0.5, 0.5)
    rot_command_clipped = np.clip(rot_command, -0.5, 0.5)
    cartesian_command = np.hstack((pos_command_clipped, rot_command_clipped))
    jacobian_p = np.zeros((3, model.nv))
    jacobian_r = np.zeros((3, model.nv))
    mujoco.mj_jacSite(model, data, jacobian_p, jacobian_r, eef_site_id)
    jacobian = np.vstack((jacobian_p, jacobian_r))
    torque_command = jacobian.T @ cartesian_command

    # 4. force compensation
    torque_compensation = data.qfrc_bias
    torque_command += torque_compensation

    # 5. apply control and render  
    data.ctrl[:] = torque_command
    mujoco.mj_step(model, data)
    viewer.render()

viewer.close()
del(viewer)

switch to down
switch to up
switch to down
switch to up
switch to down
switch to up
switch to down
switch to up
